# Lab - Observability: One Trace Across an Agent Run

Three tasks that follow customer requests through every call of a multi-agent run - a coordinator, an order agent and a billing agent - without changing a line of the agent code.

| Task | What you learn |
| --- | --- |
| 1 | Wrap messages.create once: one JSON line per call, with a trace ID |
| 2 | One trace as a table: p50, p95, cost and the slowest step |
| 3 | Across runs: calls, latency and the share of the bill per agent |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell runs offline - and, offline, at a realistic pace, so there are times to measure. The wrapper you write is exactly the one you would put on the real SDK.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import statistics
import time
import uuid
from pathlib import Path

load_dotenv()
client = Anthropic()
shopassist_lab.simulate_latency(True, per_token=0.004, first=0.25)   # offline only: a realistic pace

MODEL = "claude-sonnet-5-5"
PRICES = {"claude-sonnet-5-5": (2.00, 10.00)}   # dollars per million tokens: input, output
LOG = "runs.jsonl"
Path(LOG).write_text("")   # an empty log for this session

# --- The coordinator from the lecture: it runs unchanged. -------------------------
ORDERS = {"ORD-87654321": {"item": "lamp", "status": "delivered", "delivered_days_ago": 45, "price": 60},
          "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20, "price": 80}}
CHARGES = {"ORD-87654321": [{"amount": 60, "date": "day 1"}, {"amount": 60, "date": "day 1"}],
           "ORD-12345678": [{"amount": 80, "date": "day 3"}]}


def tool(name, description, fields):
    return {"name": name, "description": description,
            "input_schema": {"type": "object", "properties": fields, "required": list(fields)}}


ORDER_TOOLS = [tool("lookup_order", "Look up an order by its ID.", {"order_id": {"type": "string"}})]
BILLING_TOOLS = [tool("list_charges", "List the charges of an order.", {"order_id": {"type": "string"}}),
                 tool("refund_charge", "Refund one charge of an order.",
                      {"order_id": {"type": "string"}, "amount": {"type": "number"}})]
FUNCTIONS = {"lookup_order": lambda order_id: ORDERS.get(order_id, {"error": "order not found"}),
             "list_charges": lambda order_id: CHARGES.get(order_id, []),
             "refund_charge": lambda order_id, amount: {"refunded": amount, "order_id": order_id}}


def run_agent(system, tools, task):
    """One agent: its own system prompt, its own tools, its own loop."""
    messages = [{"role": "user", "content": task}]
    while True:
        response = client.messages.create(model=MODEL, max_tokens=600,
                                          output_config={"effort": "low"},
                                          system=system, tools=tools, messages=messages)
        if response.stop_reason != "tool_use":
            return next(b.text for b in response.content if b.type == "text")
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": b.id,
             "content": json.dumps(FUNCTIONS[b.name](**b.input))}
            for b in response.content if b.type == "tool_use"]})


def order_agent(order_id):
    return run_agent("You check orders. Report the item, status and delivery age in one "
                     "sentence.", ORDER_TOOLS, f"Check order {order_id}.")


def billing_agent(order_id, facts):
    return run_agent("You fix billing problems. Refund only a duplicate charge. Report what "
                     "you did in one sentence.", BILLING_TOOLS,
                     f"Order facts: {facts}\nCheck the charges of {order_id} and fix any duplicate.")


COORDINATOR_TOOLS = [
    tool("order_agent", "Delegate: check an order.", {"order_id": {"type": "string"}}),
    tool("billing_agent", "Delegate: check and fix the charges of an order. Pass the order "
         "facts you already have.", {"order_id": {"type": "string"}, "facts": {"type": "string"}}),
]
FUNCTIONS.update({"order_agent": order_agent, "billing_agent": billing_agent})


def handle(message):
    """The coordinator: no data tools, two agents to delegate to."""
    return run_agent("You are ShopAssist's coordinator. You have no data tools. Delegate to the "
                     "order agent and the billing agent, pass each one what it needs, and "
                     "answer the customer in two sentences.", COORDINATOR_TOOLS, message)

---

## Task 1 - Wrap the client once, log every call

In [ ]:
# ============================================================
# TASK 1 - Wrap the client once, log every call
# ============================================================
#
# WHAT TO DO
#   Every call to Claude goes through one function in the
#   library: messages.create. Keep the original, put yours in its
#   place: note the time, call the original, write one JSON line
#   per call. Then run the coordinator - not one line of it
#   changes.
#
# WHY IT MATTERS
#   When a coordinator talks to subagents and tools, one slow or
#   expensive call hides inside a single answer. One line per
#   call - the trace ID of the run, the agent, milliseconds,
#   tokens, cost, stop reason and tools - turns the run into
#   something you can read.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We keep the original function, and we put our own function
#   in its place".
#
# HOW TO DO IT
#   Three blanks:
#
#       original_create(self, **request)       the real call, unchanged
#       round((time.time() - start) * 1000)    milliseconds
#       json.dumps(record)                     one line of JSON per call
# ============================================================

from anthropic.resources.messages import Messages

AGENTS = {"coordinator": "coordinator", "check orders": "order", "billing": "billing"}
CURRENT = {"trace": None}   # provided: the trace ID of the run in progress
original_create = getattr(Messages, "untraced_create", Messages.create)   # provided: wrap once,
Messages.untraced_create = original_create                                 # even when re-run

try:
    # TODO: replace each ... below with the value named beside it
    def traced_create(self, **request):
        start = time.time()
        response = ...   # original_create(self, **request)
        usage = response.usage
        price_in, price_out = PRICES[request["model"]]
        record = {
            "trace": CURRENT["trace"],
            "agent": next(name for key, name in AGENTS.items() if key in request["system"]),
            "ms": ...,   # round((time.time() - start) * 1000)
            "in": usage.input_tokens, "out": usage.output_tokens,
            "cost": (usage.input_tokens * price_in + usage.output_tokens * price_out) / 1e6,
            "stop": response.stop_reason,
            "tools": [b.name for b in response.content if b.type == "tool_use"],
        }
        with open(LOG, "a") as f:
            f.write(... + "\n")   # json.dumps(record)
        return response

    Messages.create = traced_create   # provided: the swap - every client now goes through it
    CURRENT["trace"] = uuid.uuid4().hex[:8]
    answer = handle("I was charged twice for order ORD-87654321. Please fix it.")
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("ShopAssist:", answer)
print(open(LOG).read().splitlines()[-1])

check("trace_wrap", log=LOG, trace=CURRENT["trace"], answer=answer)

---

## Task 2 - One trace: every call, the tail, the slowest step

In [ ]:
# ============================================================
# TASK 2 - One trace: every call, the tail, the slowest step
# ============================================================
#
# WHAT TO DO
#   Read back the lines of this trace and print one row per
#   call. Then the median time, the 95th percentile, the total
#   cost, and the step that made this customer wait the longest.
#
# WHY IT MATTERS
#   The total tells you the customer waited; the trace tells you
#   why, and what each agent cost. The median shows a typical
#   call; the 95th percentile shows the tail - where the slow
#   calls live, and what averages hide.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The last line names the step that made this customer wait
#   the longest".
#
# HOW TO DO IT
#   Three blanks:
#
#       "trace"                              only this run's lines
#       times[round(0.95 * (len(times) - 1))]   the 95th percentile
#       max(rows, key=lambda r: r["ms"])     the slowest call
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def summary(trace_id):
        rows = [json.loads(line) for line in open(LOG) if line.strip()]
        rows = [r for r in rows if r[...] == trace_id]   # "trace"
        if not rows:   # provided: nothing logged for this run yet
            raise ValueError("No calls of this run in the log - finish Task 1 first.")
        times = sorted(r["ms"] for r in rows)
        slowest = ...   # max(rows, key=lambda r: r["ms"])
        return {"calls": len(rows), "p50_ms": statistics.median(times),
                "p95_ms": ...,   # times[round(0.95 * (len(times) - 1))]
                "cost": round(sum(r["cost"] for r in rows), 5),
                "slowest": f"{slowest['agent']}, {slowest['ms']} ms"}, rows

    stats, rows = summary(CURRENT["trace"])
    if ... in stats.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
print(f"trace {CURRENT['trace']}: {stats['calls']} calls")
for r in rows:
    print(f"  {r['agent']:12} {r['ms']:5} ms {r['in']:5} in {r['out']:4} out  {r['stop']:9} {' '.join(r['tools'])}")
print(f"p50 {stats['p50_ms']} ms  p95 {stats['p95_ms']} ms  cost ${stats['cost']}  slowest: {stats['slowest']}")

check("trace_summary", stats=stats, rows=rows)

---

## Task 3 - Across runs: what each agent costs

In [ ]:
# ============================================================
# TASK 3 - Across runs: what each agent costs
# ============================================================
#
# WHAT TO DO
#   Run the coordinator for two more customers, each with its own
#   trace ID. Then read the whole log and group it by agent:
#   calls, average milliseconds, total cost and the agent's share
#   of the bill.
#
# WHY IT MATTERS
#   With a trace ID on every line, the same log answers questions
#   across thousands of runs: which agent is slowest, which costs
#   most, which grew in tokens after a prompt change. That is
#   where an architect looks before splitting an agent, changing
#   its model or cutting its context.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Which agent is slowest, which tool fails, and which step
#   grew in tokens after a prompt change".
#
# HOW TO DO IT
#   Three blanks:
#
#       uuid.uuid4().hex[:8]        a new trace ID for every run
#       r["agent"]                  the group this line counts in
#       cost / total                the agent's share of the bill
# ============================================================

MORE = ["Was I charged twice for order ORD-12345678? Please check.",
        "I was charged twice for order ORD-87654321 and want my money back."]

try:
    # TODO: replace each ... below with the value named beside it
    for message in MORE:
        CURRENT["trace"] = ...   # uuid.uuid4().hex[:8]
        handle(message)
    
    log = [json.loads(line) for line in open(LOG) if line.strip()]
    total = sum(r["cost"] for r in log)
    by_agent = {}
    for r in log:
        group = by_agent.setdefault(..., {"calls": 0, "ms": 0, "cost": 0.0})   # r["agent"]
        group["calls"] += 1
        group["ms"] += r["ms"]
        group["cost"] += r["cost"]
    for name, group in by_agent.items():
        group["avg_ms"] = round(group.pop("ms") / group["calls"])
        group["share"] = round(..., 2)   # group["cost"] / total
        group["cost"] = round(group["cost"], 5)
    if ... in by_agent:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
print(f"{len({r['trace'] for r in log})} runs, {len(log)} calls")
for name, group in sorted(by_agent.items(), key=lambda kv: -kv[1]["cost"]):
    print(f"  {name:12} {group['calls']:3} calls  avg {group['avg_ms']:5} ms  ${group['cost']:.5f}"
          f"  {group['share']:.0%} of the bill")

check("trace_agents", by_agent=by_agent, log=log)

---

## Done

Wrap the model client once, log every call with a trace ID, the agent, latency, tokens, cost and stop reason, and look at the 95th percentile, the slowest step and the cost per agent - not only at the average.